In [1]:
!nvidia-smi

Sat May 23 08:22:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install colabcode
!pip install fastapi

Requested uvicorn==0.13.1 from https://files.pythonhosted.org/packages/ef/67/546c35e9fffb585ea0608ba3bdcafe17ae402e304367203d0b08d6c23051/uvicorn-0.13.1-py3-none-any.whl (from colabcode) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    python-dotenv (>=0.13.*) ; extra == 'standard'
                   ~~~~~~~^
Please use pip<24.1 if you need to use this version.
INFO: pip is looking at multiple versions of colabcode to determine which version is compatible with other requirements. This could take a while.
  Using cached uvicorn-0.13.1-py3-none-any.whl.metadata (4.6 kB)
Requested uvicorn==0.13.1 from https://files.pythonhosted.org/packages/ef/67/546c35e9fffb585ea0608ba3bdcafe17ae402e304367203d0b08d6c23051/uvicorn-0.13.1-py3-none-any.whl (from colabcode) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    python-dotenv (>=0.13.*) ; extra == 'standard'
                   ~~~~~~~^
Please use pip<24.1 if you need to use this versio

In [3]:
!git clone https://github.com/stylegan-human/StyleGAN-Human.git 
# !git clone https://github.com/Lakshmanaraja/StyleGAN-Human.git

Cloning into 'StyleGAN-Human'...
remote: Enumerating objects: 365, done.
remote: Counting objects: 100% (365/365), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 365 (delta 124), reused 301 (delta 98), pack-reused 0 (from 0)
Receiving objects: 100% (365/365), 74.85 MiB | 42.63 MiB/s, done.
Resolving deltas: 100% (124/124), done.


In [4]:
!ls /content/StyleGAN-Human/*.py

/content/StyleGAN-Human/alignment.py
/content/StyleGAN-Human/bg_white.py
/content/StyleGAN-Human/edit.py
/content/StyleGAN-Human/generate.py
/content/StyleGAN-Human/insetgan.py
/content/StyleGAN-Human/interpolation.py
/content/StyleGAN-Human/legacy.py
/content/StyleGAN-Human/run_pti.py
/content/StyleGAN-Human/style_mixing.py
/content/StyleGAN-Human/stylemixing_video.py


In [5]:
# Install ninja (required for JIT compilation of CUDA extensions)
!pip install ninja -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 9.9 MB/s eta 0:00:00


In [6]:
!pip install lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.8 MB/s eta 0:00:00


In [7]:
import pathlib, os, shutil

# 1. Clear stale CUDA extension cache from previous Colab sessions
cache = os.path.expanduser('~/.cache/torch_extensions')
if os.path.exists(cache):
    shutil.rmtree(cache)
    print('Cleared torch_extensions cache')

# 2. Fix custom_ops.py for PyTorch 2.x:
#    torch.utils.cpp_extension.load() no longer registers the module in sys.modules,
#    so importlib.import_module() raises 'No module named upfirdn2d_plugin'.
#    Fix: use the return value of load() directly.
for p in pathlib.Path('.').rglob('torch_utils/custom_ops.py'):
    text = p.read_text()
    fixed = text
    fixed = fixed.replace(
        '            torch.utils.cpp_extension.load(name=module_name, build_directory=build_dir,\n'
        '                verbose=verbose_build, sources=digest_sources, **build_kwargs)\n'
        '        else:\n'
        '            torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)\n'
        '        module = importlib.import_module(module_name)',
        '            module = torch.utils.cpp_extension.load(name=module_name, build_directory=build_dir,\n'
        '                verbose=verbose_build, sources=digest_sources, **build_kwargs)\n'
        '        else:\n'
        '            module = torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)'
    )
    fixed = fixed.replace(
        '            torch.utils.cpp_extension.load(name=module_name, build_directory=cached_build_dir,\n'
        '                verbose=verbose_build, sources=cached_sources, **build_kwargs)\n'
        '        else:\n'
        '            torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)\n'
        '\n'
        '        # Load.\n'
        '        module = importlib.import_module(module_name)',
        '            module = torch.utils.cpp_extension.load(name=module_name, build_directory=cached_build_dir,\n'
        '                verbose=verbose_build, sources=cached_sources, **build_kwargs)\n'
        '        else:\n'
        '            module = torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)'
    )
    if fixed != text:
        p.write_text(fixed)
        print(f'Patched custom_ops.py: {p}')
    else:
        print(f'Already patched or pattern mismatch: {p}')

# 3. Fix PIL.Image.ANTIALIAS -> LANCZOS (removed in Pillow 10.0)
for p in pathlib.Path('.').rglob('*.py'):
    try:
        text = p.read_text()
        if 'ANTIALIAS' in text:
            p.write_text(text.replace('PIL.Image.ANTIALIAS', 'PIL.Image.LANCZOS')
                            .replace('Image.ANTIALIAS', 'Image.LANCZOS'))
            print(f'Fixed ANTIALIAS: {p}')
    except Exception:
        pass

# 4. Fix op_edit C++ sources for PyTorch 2.x:
#    - Tensor::type().is_cuda() removed -> use is_cuda() directly
#    - ATen/cuda/CUDAApplyUtils.cuh removed in PyTorch 2.0
for p in pathlib.Path('.').rglob('torch_utils/op_edit/*.cpp'):
    text = p.read_text()
    fixed = text.replace('x.type().is_cuda()', 'x.is_cuda()')
    if fixed != text:
        p.write_text(fixed)
        print(f'Fixed Tensor::type() in: {p}')

for p in pathlib.Path('.').rglob('torch_utils/op_edit/*.cu'):
    lines = p.read_text().splitlines(keepends=True)
    clean = [l for l in lines if 'CUDAApplyUtils.cuh' not in l]
    if len(clean) != len(lines):
        p.write_text(''.join(clean))
        print(f'Removed CUDAApplyUtils.cuh from: {p}')

print('Done.')


Patched custom_ops.py: StyleGAN-Human/torch_utils/custom_ops.py
Fixed ANTIALIAS: StyleGAN-Human/utils/face_alignment.py
Fixed Tensor::type() in: StyleGAN-Human/torch_utils/op_edit/upfirdn2d.cpp
Fixed Tensor::type() in: StyleGAN-Human/torch_utils/op_edit/fused_bias_act.cpp
Removed CUDAApplyUtils.cuh from: StyleGAN-Human/torch_utils/op_edit/fused_bias_act_kernel.cu
Removed CUDAApplyUtils.cuh from: StyleGAN-Human/torch_utils/op_edit/upfirdn2d_kernel.cu
Done.


## Compatibility fixes (PyTorch 2.x + Pillow 10.x)
Run once after cloning. Patches two upstream bugs:
1. **`custom_ops.py`** — `importlib.import_module()` fails after `torch.utils.cpp_extension.load()` in PyTorch 2.x (`No module named 'upfirdn2d_plugin'`). Fixed by using the return value of `load()` directly.
2. **`PIL.Image.ANTIALIAS`** — removed in Pillow 10.0, replaced with `PIL.Image.LANCZOS`.

In [8]:
import os
repo_name = 'StyleGAN-Human'
os.chdir(f'./{repo_name}')

In [9]:
!ngrok authtoken 29LR1RD7sCbaYhliJQGaohZWYud_5A5g8831CiiBnNR2yvvra

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml                                


In [10]:
def get_download_model_command(file_id, file_name):
    """Download a model from Google Drive using gdown and save to pretrained_models/."""
    current_directory = os.getcwd()
    save_path = os.path.join(os.path.dirname(current_directory), f'{repo_name}', "pretrained_models")
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    # &confirm=t bypasses the Google Drive virus-scan warning page
    url = f'gdown "https://drive.google.com/uc?id={file_id}&confirm=t" -O "{save_path}/{file_name}"'
    return url

In [11]:
MODEL_PATHS = {
    "stylegan1_1024": {"id": "1h-R-IV-INGdPEzj4P9ml6JTEvihuNgLX", "name": "stylegan1_1024.pkl"},
    "stylegan2_1024": {"id": "1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5", "name": "stylegan2_1024.pkl"},
    "stylegan2_512": {"id": "1dlFEHbu-WzQWJl7nBBZYcTyo000H9hVm", "name": "stylegan2_512.pkl"},
    "stylegan3_512": {"id": "1_274jk_N6WSCkKWeu7hjHycqGvbuOFf5", "name": "stylegan3_512.pkl"},
    # "stylegan3_1024": {"id": None, "name": "stylegan3_1024.pkl"},
    # "stylegan1_512": {"id": None, "name": "stylegan1_512.pkl"},
}

In [ ]:
#@title Select which experiment you wish to perform inference on: { run: "auto" }
experiment_type = 'stylegan2_1024' #@param ['stylegan1_1024', 'stylegan2_1024', 'stylegan1_512', 'stylegan2_512', 'stylegan3_512']  

In [13]:
path = MODEL_PATHS[experiment_type]
download_command = get_download_model_command(file_id=path["id"], file_name=path["name"])
!{download_command}

Downloading...
From: https://drive.google.com/uc?id=1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5&confirm=t
To: /content/StyleGAN-Human/pretrained_models/stylegan2_1024.pkl
100% 362M/362M [00:05<00:00, 61.5MB/s] 


In [14]:
version=experiment_type.split("_")[0][-1] 
version

'2'

In [15]:
## Download pretrained StyleGAN on FFHQ 1024x1024 and dlib models.

# ffhq.pkl: downloaded directly from NVIDIA CDN (more reliable than Google Drive)
current_directory = os.getcwd()
ffhq_save_path = os.path.join(os.path.dirname(current_directory), f'{repo_name}', "pretrained_models", "ffhq.pkl")
!wget -q --show-progress \
    https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl \
    -O {ffhq_save_path}

# dlib face detector and landmark predictor
dlib_detector = get_download_model_command(file_id="1MduBgju5KFNrQfDLoQXJ_1_h5MnctCIG", file_name='mmod_human_face_detector.dat')
dlib_landmark = get_download_model_command(file_id="1A82DnJBJzt8wI2J8ZrCK5fgHcQ2-tcWM", file_name='shape_predictor_68_face_landmarks.dat')
!{dlib_detector}
!{dlib_landmark}

/content/StyleGAN-H 100%[===================>] 363.94M  81.1MB/s    in 4.5s    
Downloading...
From: https://drive.google.com/uc?id=1MduBgju5KFNrQfDLoQXJ_1_h5MnctCIG&confirm=t
To: /content/StyleGAN-Human/pretrained_models/mmod_human_face_detector.dat
100% 730k/730k [00:00<00:00, 149MB/s]
Downloading...
From: https://drive.google.com/uc?id=1A82DnJBJzt8wI2J8ZrCK5fgHcQ2-tcWM&confirm=t
To: /content/StyleGAN-Human/pretrained_models/shape_predictor_68_face_landmarks.dat
100% 99.7M/99.7M [00:02<00:00, 45.0MB/s]


In [16]:
#!python generate.py --outdir=outputs/{experiment_type}/ --seeds=12-15 --trunc=0.2 --network=pretrained_models/{experiment_type}.pkl --version {version}

In [ ]:
#!python edit.py --outdir outputs/editing --network pretrained_models/stylegan2_1024.pkl --attr_name upper_length --seeds 61531,61570,61571,61610 


In [18]:
#!python style_mixing.py --outdir=outputs/stylemixing --rows=85,100,75,458,1500,86 --cols=55,821,1789,293,75 --network=pretrained_models/stylegan2_1024.pkl --styles=0-3  


In [19]:
# Perform joint optimization and generate seamless images
#!python insetgan.py --face_seed=9 --body_seed=89180 \
                   # --joint_steps=500 --outdir outputs/insetgan --video 1

# **Sending ZIP File as Response**

In [20]:
import os
import zipfile as zf 
import io
from io import BytesIO
from fastapi.responses import StreamingResponse

def zipfile_new(file_list):
    zip_io = BytesIO()
    zip_sub_dir = "final_archive"
    zip_filename = "%s.zip" % zip_sub_dir
    with zf.ZipFile(zip_io, mode='w', compression=zf.ZIP_DEFLATED) as zip:
        for fpath in file_list:
            zip.write(fpath)
        #close zip
        zip.close()
    return StreamingResponse(
        iter([zip_io.getvalue()]),
        media_type="application/x-zip-compressed",
        headers = { "Content-Disposition":f"attachment;filename=%s" % zip_filename}
    )

In [ ]:
import os, sys
repo_dir = '/content/StyleGAN-Human'
if os.getcwd() != repo_dir:
    os.chdir(repo_dir)
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

In [22]:
import os; print(os.getcwd())

/content/StyleGAN-Human


In [23]:
import pathlib

for src_name, func_to_expose in [
    ('generate.py', None),
    ('edit.py', None),
    ('style_mixing.py', None),
    ('insetgan.py', None),
]:
    src = pathlib.Path(f'/content/StyleGAN-Human/{src_name}')
    dst = pathlib.Path(f'/content/StyleGAN-Human/{src_name.replace(".py", "1.py")}')
    if dst.exists():
        print(f'{dst.name} already exists, skipping')
        continue
    text = src.read_text()
    # Remove @click decorators so functions are plain callables
    import re
    text = re.sub(r'@click\.[^\n]+\n', '', text)
    text = text.replace('    ctx: click.Context,\n', '')
    text = text.replace('ctx: click.Context,\n', '')
    dst.write_text(text)
    print(f'Created {dst.name}')


Created generate1.py
Created edit1.py
Created style_mixing1.py
Created insetgan1.py


# **API END POINTS**

In [24]:
from generate1 import generate_images
from edit1 import main as editmain
from style_mixing1 import generate_style_mix
from insetgan1 import main as insmain
from fastapi import FastAPI
from fastapi.responses import FileResponse


import legacy
app = FastAPI()

#path = "/path/to/files"

path = '/content/StyleGAN-Human/outputs/stylegan2_1024/'
#cmd = '!python generate.py --outdir=outputs/{experiment_type}/ --seeds=13 --trunc=0.2 --network=pretrained_models/{experiment_type}.pkl --version {version}'

@app.get("/")
def index():
    return {"Hello": "World"}

@app.get("/generate_single_image" ) #, responses={200: {"description": "A picture of a vector image.", "content" : {"image/jpeg" : {"example" : "No example available. Just imagine a picture of a vector image."}}}})
def generate_single_image_endpoint(seed="12",trunc=0.5):
  print(seed)
    #Generate Image here
  generate_images(
    #ctx = click.Context,
    network_pkl= 'pretrained_models/stylegan2_1024.pkl',
    seeds = legacy.num_range(seed),  # seeds = legacy.num_range("21-23"),
    truncation_psi = float(trunc), #trunc,
    outdir = '/content/StyleGAN-Human/outputs/stylegan2_1024',
    noise_mode = 'const',
    version = 2 )

    #Generate Image here Ends
  
  for seed_idx, seed_val in enumerate(legacy.num_range(seed)):
    file_name = f'seed{seed_val:04d}.png' 
     #seed00"+seed_val+".png"
    file_path = os.path.join(path, file_name )
    print(file_path)
    if os.path.exists(file_path):
       return FileResponse(file_path, media_type="image/png", filename=file_name)
    return {"error" : "File not found!"}

@app.get("/generate_image" ) #, responses={200: {"description": "A picture of a vector image.", "content" : {"image/jpeg" : {"example" : "No example available. Just imagine a picture of a vector image."}}}})
def generate_images_endpoint(seed="12",trunc=0.5):
  print(seed)
    #Generate Image here
  file_paths = list(legacy.num_range(seed))
  path = '/content/StyleGAN-Human/outputs/stylegan2_1024/'
  generate_images(
    #ctx = click.Context,
    network_pkl= 'pretrained_models/stylegan2_1024.pkl',
    seeds = legacy.num_range(seed),  # seeds = legacy.num_range("21-23"),
    truncation_psi = float(trunc), #trunc,
    outdir = '/content/StyleGAN-Human/outputs/stylegan2_1024',
    noise_mode = 'const',
    version = 2 )

  for seed_idx, seed_val in enumerate(legacy.num_range(seed)):
       file_name = f'seed{seed_val:04d}.png' 
       file_paths[seed_idx] = os.path.join(path, file_name )

  return( zipfile_new(file_paths) )

@app.get("/upper_length_Edit" ) #, responses={200: {"description": "A picture of a vector image.", "content" : {"image/jpeg" : {"example" : "No example available. Just imagine a picture of a vector image."}}}})
def upper_length_editing_endpoint(seed):
    print(seed)
    seed_list = list(seed.split(','))
    seed_list = list(map(int, seed_list))
    print(seed_list)
    print(len(seed_list))
    file_paths = list(seed_list)
                 
    path = '/content/StyleGAN-Human/outputs/editing/video'
    editmain( ckpt_path= "pretrained_models/stylegan2_1024.pkl" , 
      attr_name = "upper_length", 
      truncation = float(1),
      gen_video = True ,
      combine = True ,
      seeds= seed_list ,
      outdir= "outputs/editing" )
    
    for i in range(len(seed_list)):
       file_name = f'upper_length_{seed_list[i]:05d}.mp4' 
       file_paths[i] = os.path.join(path, file_name )

    return( zipfile_new(file_paths) )

@app.get("/bottom_length_Edit" ) #, responses={200: {"description": "A picture of a vector image.", "content" : {"image/jpeg" : {"example" : "No example available. Just imagine a picture of a vector image."}}}})
def bottom_length_editing_endpoint(seed):
    print(seed)
    seed_list = list(seed.split(','))
    seed_list = list(map(int, seed_list))
    print(seed_list)
    print(len(seed_list))
    file_paths = list(seed_list)
                 
    path = '/content/StyleGAN-Human/outputs/editing/video'
    editmain( ckpt_path= "pretrained_models/stylegan2_1024.pkl" , 
      attr_name = "bottom_length", 
      truncation = float(1),
      gen_video = True ,
      combine = True ,
      seeds= seed_list ,
      outdir= "outputs/editing" )
    
    for i in range(len(seed_list)):
       file_name = f'bottom_length_{seed_list[i]:05d}.mp4' 
       file_paths[i] = os.path.join(path, file_name )

    return( zipfile_new(file_paths) )

@app.get("/Style_Mixing_EndPoint" )
def style_mixing_endpoint (row_seeds = '85,100,75,458,1500,86' , 
                           col_seeds = '55,821,1789,293,75' , col_styles ='0,1,2,3' , trunc = 1.0  ):

    row_seed_list = list(row_seeds.split(','))
    row_seed_list = list(map(int, row_seed_list))

    col_seed_list = list(row_seeds.split(','))
    col_seed_list = list(map(int, col_seed_list))

    style_seed_list = list(col_styles.split(','))
    style_seed_list = list(map(int, style_seed_list))

    truncation_psi = float(trunc)

    path = '/content/StyleGAN-Human/outputs/stylemixing/'
  
    file_paths = ['']
    print(len(file_paths))

    generate_style_mix( 
         network_pkl = 'pretrained_models/stylegan2_1024.pkl',
         row_seeds = row_seed_list,
         col_seeds = col_seed_list,
         col_styles = style_seed_list,
         truncation_psi=truncation_psi,
         noise_mode='const',
         outdir='outputs/stylemixing'
        )
    file_name = 'grid.png' 
    file_paths[0] = os.path.join(path, file_name )
    return( zipfile_new(file_paths) )


@app.get("/Joint_Optimisation_Endpoint" )
def joint_optimisation_endpoint (face_seed = '9', body_seed = '89180' , joint_steps= '30' ,trunc ='1' ):
    face_network = "pretrained_models/ffhq.pkl"
    body_network = "./pretrained_models/stylegan2_1024.pkl"
    face_seed_int = int(face_seed)
    body_seed_int = int(body_seed)
    joint_steps_int = int(joint_steps)
    trunc_float = float(trunc)

    insmain(
        face_network = face_network,
        body_network = body_network,
        face_seed = face_seed_int,
        body_seed = body_seed_int,
        joint_steps= joint_steps_int ,
        truncation_psi = trunc_float ,
        outdir = 'outputs/insetgan',
        video = 1)
    
    path = '/content/StyleGAN-Human/outputs/insetgan/'
    file_paths=['','']
    file_name_png = f'{face_seed_int:04d}_{body_seed_int:04d}.png'
    file_paths[0] = os.path.join(path, file_name_png )
    file_name_mp4 = f'{face_seed_int:04d}_{body_seed_int:04d}.mp4' 
    file_paths[1] = os.path.join(path, file_name_mp4 )
       

    return(zipfile_new(file_paths))

/content/StyleGAN-Human/legacy.py:154: SyntaxWarning: invalid escape sequence '\d'
  n_mapping =  max([int(re.findall("(\d+)", n)[0]) for n in mapping_names]) + 1
/content/StyleGAN-Human/legacy.py:155: SyntaxWarning: invalid escape sequence '\d'
  resolution =  max([int(re.findall("(\d+)", n)[0]) for n in sythesis_names])


# **Running the APP**

In [25]:
from pyngrok import ngrok
import uvicorn, threading

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")

# Run FastAPI app in background thread
thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000),
    daemon=True
)
thread.start()


Public URL: NgrokTunnel: "https://43b0-34-34-121-101.ngrok-free.app" -> "http://localhost:8000"


# **Other references*

In [26]:
# import os
# from generate1 import generate_images
# from zipfile import ZipFile

# seed = '45-50'
# trunc = '0.7'
# file_paths = list(legacy.num_range(seed))
# path = '/content/StyleGAN-Human/outputs/stylegan2_1024/'
# generate_images(
#     #ctx = click.Context,
#     network_pkl= 'pretrained_models/stylegan2_1024.pkl',
#     seeds = legacy.num_range(seed),  # seeds = legacy.num_range("21-23"),
#     truncation_psi = float(trunc), #trunc,
#     outdir = '/content/StyleGAN-Human/outputs/stylegan2_1024',
#     noise_mode = 'const',
#     version = 2 )

# for seed_idx, seed_val in enumerate(legacy.num_range(seed)):
#        file_name = f'seed{seed_val:04d}.png' 
#        file_paths[seed_idx] = os.path.join(path, file_name )

# return( zipfile_new(file_paths) )

INFO:     Started server process [667]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [27]:
#  from generate1 import generate_images

#  generate_images(
#     #ctx = click.Context,
#     network_pkl= 'pretrained_models/stylegan2_1024.pkl',
#     seeds = legacy.num_range("12"),  # seeds = legacy.num_range("21-23"),
#     truncation_psi = float(0.7), #trunc,
#     outdir = '/content/StyleGAN-Human/outputs/stylegan2_1024',
#     noise_mode = 'const',
#     version = 2 )

In [28]:
# import edit1 
# import legacy
# app = FastAPI()

# def upper_length_editing_endpoint(seed=[61531,61570,61571,61610]):
#     file_paths = len(seed)
#     path = '/content/StyleGAN-Human/outputs/Editing/video'
#     edit1.main( ckpt_path= "pretrained_models/stylegan2_1024.pkl" , 
#       attr_name = "upper_length", 
#       truncation = float(1),
#       gen_video = True ,
#       combine = True ,
#       seeds= seed ,
#       outdir= "outputs/editing" )
    
#     for seed_idx, seed_val in enumerate(len(seed)):
#        file_name = f'upper_length_{seed_val:04d}.mp4' 
#        file_paths[seed_idx] = os.path.join(path, file_name )

#     return( zipfile_new(file_paths) )

In [29]:
# from generate1 import generate_images
# import edit1
# import legacy

# def upper_length_editing_endpoint(seed):
#     print(seed)
#     seed_list = list(seed.split(','))
#     seed_list = list(map(int, seed_list))
#     print(seed_list)
#     print(len(seed_list))
#     file_paths = list(seed_list)

#     path = '/content/StyleGAN-Human/outputs/editing/video/'
#     edit1.main( ckpt_path= "pretrained_models/stylegan2_1024.pkl" , 
#       attr_name = "upper_length", 
#       truncation = float(1),
#       gen_video = True ,
#       combine = True ,
#       seeds= seed_list ,
#       outdir= "outputs/editing" )
    
#     for i in range(len(seed_list)):
#        file_name = f'upper_length_{seed_list[i]:05d}.mp4' 
#        file_paths[i] = os.path.join(path, file_name )

#     zipfile_new(file_paths) 

In [30]:
#upper_length_editing_endpoint("20,30,40,1000")

#/content/StyleGAN-Human/outputs/editing/video/upper_length_00020.mp4
#/content/StyleGAN-Human/outputs/editing/video/upper_length_00020.mp4

In [31]:
# from style_mixing1 import generate_style_mix

# def style_mixing_endpoint (row_seeds = '85,100,75,458,1500,86' , 
#                            col_seeds = '55,821,1789,293,75' , col_styles ='0,1,2,3' , trunc = 1.0  ):

#     row_seed_list = list(row_seeds.split(','))
#     row_seed_list = list(map(int, row_seed_list))

#     col_seed_list = list(row_seeds.split(','))
#     col_seed_list = list(map(int, col_seed_list))

#     style_seed_list = list(col_styles.split(','))
#     style_seed_list = list(map(int, style_seed_list))

#     truncation_psi = trunc

#     path = '/content/StyleGAN-Human/outputs/stylegan2_1024/'

#     file_paths = ['']
#     print(len(file_paths))

#     generate_style_mix( 
#          network_pkl = 'pretrained_models/stylegan2_1024.pkl',
#          row_seeds = row_seed_list,
#          col_seeds = col_seed_list,
#          col_styles = style_seed_list,
#          truncation_psi=truncation_psi,
#          noise_mode='const',
#          outdir='outputs/stylemixing'
#         )
#     file_name = 'grid.png' 
#     file_paths[0] = os.path.join(path, file_name )
#     return( zipfile_new(file_paths) )

In [32]:
##style_mixing_endpoint()

In [33]:
# import os
# import legacy
# from fastapi import FastAPI 
# from fastapi.responses import FileResponse
# from generate1 import generate_images

# app = FastAPI()

# #path = "/path/to/files"

# path = '/content/StyleGAN-Human/outputs/stylegan2_1024/'
# #cmd = '!python generate.py --outdir=outputs/{experiment_type}/ --seeds=13 --trunc=0.2 --network=pretrained_models/{experiment_type}.pkl --version {version}'

# @app.get("/")
# def index():
#     return {"Hello": "World"}

# @app.get("/vector_image", responses={200: {"description": "A picture of a vector image.", "content" : {"image/jpeg" : {"example" : "No example available. Just imagine a picture of a vector image."}}}})
# def image_endpoint(seed="12",trunc=0.5):
#   print(seed)
#     #Generate Image here
#   generate_images(
#     #ctx = click.Context,
#     network_pkl= 'pretrained_models/stylegan2_1024.pkl',
#     seeds = legacy.num_range(seed),  # seeds = legacy.num_range("21-23"),
#     truncation_psi = float(trunc), #trunc,
#     outdir = '/content/StyleGAN-Human/outputs/stylegan2_1024',
#     noise_mode = 'const',
#     version = 2 )

#     #Generate Image here Ends
  
#   for seed_idx, seed_val in enumerate(legacy.num_range(seed)):
#     file_name = f'seed{seed_val:04d}.png' 
#      #seed00"+seed_val+".png"
#     file_path = os.path.join(path, file_name )
#     print(file_path)
#     if os.path.exists(file_path):
#        return FileResponse(file_path, media_type="image/png", filename=file_name)
#     return {"error" : "File not found!"}

In [34]:
# from insetgan1 import main

# def inset_gan_joint_optimisation(face_seed = '9', body_seed = '89180' , joint_steps= '30' ,trunc ='1' ):

#     face_network = "pretrained_models/ffhq.pkl"
#     body_network = "./pretrained_models/stylegan2_1024.pkl"
#     face_seed_int = int(face_seed)
#     body_seed_int = int(body_seed)
#     joint_steps_int = int(joint_steps)
#     trunc_float = float(trunc)

#     main(
#         face_network = face_network,
#         body_network = body_network,
#         face_seed = face_seed_int,
#         body_seed = body_seed_int,
#         joint_steps= joint_steps_int ,
#         truncation_psi = trunc_float ,
#         outdir = 'outputs/insetgan',
#         video = 1)
    
#     path = '/content/StyleGAN-Human/outputs/insetgan/'
#     file_paths=['','']
#     file_name_png = f'{face_seed_int:04d}_{body_seed_int:04d}.png'
#     file_paths[0] = os.path.join(path, file_name_png )
#     file_name_mp4 = f'{face_seed_int:04d}_{body_seed_int:04d}.mp4' 
#     file_paths[1] = os.path.join(path, file_name_mp4 )
       

#     return( zipfile_new(file_paths) )

In [35]:
#inset_gan_joint_optimisation('20','20345')